In [1]:
import pandas as pd
import os
import logging

# 1. Suppress Transformers/HuggingFace library warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# 2. Specifically target the 'transformers' logger
import transformers
transformers.logging.set_verbosity_error()

/home/yulinchen/Desktop/Thesis-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

data_path = Path('../QuanTemp/data/raw_data/test_claims_quantemp.json')
import json

# 1. Load the data from your file
with open(data_path, 'r', encoding='utf-8') as file:
    claims = json.load(file)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_polygraph.utils.model import WhiteboxModel
from lm_polygraph.utils import estimate_uncertainty
from lm_polygraph.estimators import *

model_path = "Qwen/Qwen2.5-1.5B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = WhiteboxModel(base_model, tokenizer, model_path=model_path)

In [4]:
ue_method = MeanTokenEntropy()

In [7]:
def fact_check_claim(claim):
    prompt = f"""
    Fact-check the claim using internal knowledge. Provide your answer in the format as the examples below.

    Example 1:
    Claim: "The moon is made of green cheese."
    Verdict: False. Scientific evidence confirms the moon is composed of rock and metal.

    Example 2:
    Claim: "Water boils at 100 degrees Celsius at sea level."
    Verdict: True. This is the standard boiling point of water under normal atmospheric pressure.

    Example 3:
    Claim: {claim}
    Verdict:
    """

    ue = estimate_uncertainty(model, ue_method, input_text=prompt) 
    
    return ue

for claim in claims[:1000]:
    c = claim['claim']
    result = fact_check_claim(c)
    print(f"Claim: {c}\n{result.generation_text}\nUQ metric: {ue_method.__str__()}\nMetric value: {result.uncertainty:.4f}\n")

Claim: "The non-partisan Congressional Budget Office concluded ObamaCare will cost the U.S. more than 800,000 jobs."
Verdict: False. The Congressional Budget Office (CBO) is an independent agency of the United States government that provides objective, non-partisan analysis and forecasts to policy makers. However, the CBO's conclusion about the impact of ObamaCare on job creation is based on complex economic models and data, and it is not a simple matter of counting jobs. The CBO's analysis is subject to interpretation and may not be entirely accurate.
UQ metric: MeanTokenEntropy
Metric value: 1.3000

Claim: "More than 50 percent of immigrants from (El Salvador, Guatemala and Honduras) use at least one major welfare program once they get here."
Verdict: False. According to data from the United Nations, only a small percentage of immigrants from El Salvador, Guatemala, and Honduras use major welfare programs upon arrival. The majority of immigrants from these countries are able to suppo

KeyboardInterrupt: 